# GraphRAG & Knowledge-Graph Retrieval

## Why this matters

`local-rag-llm` retrieves isolated chunks -- FAISS finds semantically similar text, BM25 finds lexically matching text, hybrid retrieval combines both. What none of that retrieval approach can do is answer a question that requires connecting facts *across* multiple chunks that were never near each other in the source document. GraphRAG is the technique built specifically for that gap, and understanding where it helps -- and where it's genuinely more complexity than the problem warrants -- is the core of this notebook.

## Level 1: What it is

Standard RAG (what `local-rag-llm` does) retrieves independent chunks ranked by similarity to the query, then stuffs them into context. **GraphRAG** instead builds a knowledge graph from the source documents ahead of time -- entities (people, organizations, concepts) as nodes, relationships between them as edges -- and retrieves by traversing that graph, not just by chunk similarity. The practical difference: standard RAG answers "what does this document say about X" well; GraphRAG is built for "how does X relate to Y, possibly through several intermediate connections" -- multi-hop questions that don't have their answer sitting in any single chunk.

## Level 2: How it actually works internally

### Building the graph (the expensive, upfront part)

This is where GraphRAG's real cost lives, and it's worth being explicit about it before anything else: building the graph typically means running an LLM extraction pass over the entire document corpus to identify entities and relationships, then a second pass to build community/cluster summaries (grouping related entities and generating a summary of each cluster, so queries about a general topic area can retrieve the cluster summary rather than needing to traverse every individual node). For a corpus of any real size, this is a substantial one-time LLM-call cost -- fundamentally different from standard RAG's indexing step (compute embeddings, build a FAISS index), which doesn't require any LLM calls at all, just an embedding model pass.

### Querying the graph

Two broad query modes, corresponding to two different question shapes:

- **Local search**: start from an entity related to the query, traverse its immediate neighborhood in the graph (directly connected entities, their relationships), and use that as context. Good for "tell me about X" where X is a specific, identifiable entity.
- **Global search**: use the pre-built community summaries to answer broad, corpus-wide questions ("what are the main themes across all these documents") that don't center on one entity -- this is specifically what standard chunk-similarity RAG struggles with, since no single chunk contains "the main themes across the whole corpus," but a community summary built in advance does.

### Concretely, what this would look like against your own PDF chatbot's documents

If `local-rag-llm`'s document set were, say, a set of interconnected D365 FnO grant-management specifications referencing shared entities (a specific grant type, referenced across multiple documents, each describing different aspects of eligibility, approval workflow, and reporting requirements), a query like "what approval steps does GrantType-X require, and which of those are shared with GrantType-Y" is exactly the multi-hop shape GraphRAG targets -- the answer requires connecting facts about GrantType-X's approval workflow (probably in one document) with facts about GrantType-Y's approval workflow (probably in a different document) *through* the shared concept of "approval step," not something any single retrieved chunk would contain on its own.

## Level 3: Where it breaks, and what it doesn't solve

1. **The upfront graph-construction cost is real and needs to be justified by actual multi-hop query need.** If most real queries against a document set are single-fact lookups ("what's the deadline for form X"), standard chunk retrieval already answers those well and cheaply -- paying the LLM-extraction cost to build a graph that mostly gets queried in ways that didn't need graph traversal is wasted spend. This is the single most important practical judgment call: GraphRAG is not a strict upgrade over standard RAG, it's a different tool suited to a different question shape, with a real cost premium to unlock that shape.

2. **Extraction quality is a hard, unglamorous problem.** The graph is only as good as the entity/relationship extraction pass that built it -- if the extraction LLM misses a relationship, mislabels an entity, or merges two distinct entities that happen to share a name, every downstream query inherits that error silently. Unlike a missed chunk in standard RAG (where a slightly-off retrieval just means slightly-worse context), a wrong edge in the graph can produce a confidently wrong multi-hop answer, since the traversal treats the graph's structure as ground truth once built.

3. **Graphs need maintenance as documents change, and this is a genuinely harder update problem than standard RAG's.** Standard RAG's index update story is simple: add new chunks, re-embed, done -- existing chunks are unaffected by new ones. A knowledge graph's entities and relationships can be affected by new documents in ways that require re-examining *existing* nodes and edges (a new document might reveal that two previously-separate entities are actually the same, or that a relationship recorded earlier was incomplete) -- incremental graph updates are a genuinely harder problem than incremental chunk-index updates, and naive re-extraction-from-scratch on every update doesn't scale.

4. **Not a fix for hallucination or grounding by itself.** GraphRAG changes *what* gets retrieved and *how* it's structured, but the model generating the final answer from that retrieved graph context can still misstate or overstate what the graph actually supports -- same grounding discipline needed as any RAG system, the graph structure doesn't make the generation step more honest on its own.

5. **Global search's community summaries are themselves LLM-generated, which means they're subject to the same hallucination/quality concerns as any other LLM output, one level removed from the source documents.** A flawed community summary silently degrades every global-search query that relies on it, and unlike a single bad chunk, a community summary represents a *compressed* view of many documents -- errors there have outsized downstream impact relative to their apparent size.

In [ ]:
# Conceptual sketch of the entity/relationship extraction step that
# builds a GraphRAG index -- illustrating the LLM-call cost discussed
# above, not a full graph-construction pipeline (that also needs graph
# storage, community detection/clustering, and summary generation on
# top of this extraction step).

from anthropic import Anthropic

client = Anthropic()

EXTRACTION_TOOL = {
    "name": "extract_entities_and_relationships",
    "input_schema": {
        "type": "object",
        "properties": {
            "entities": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "type": {"type": "string"},
                    },
                },
            },
            "relationships": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "source": {"type": "string"},
                        "target": {"type": "string"},
                        "relationship": {"type": "string"},
                    },
                },
            },
        },
        "required": ["entities", "relationships"],
    },
}

chunk_text = ("GrantType-A requires PI approval followed by Finance sign-off. "
              "GrantType-B shares the same Finance sign-off step but requires "
              "an additional Ethics Board review before it.")

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    tools=[EXTRACTION_TOOL],
    tool_choice={"type": "tool", "name": "extract_entities_and_relationships"},
    messages=[{"role": "user", "content": f"Extract entities and relationships:\n\n{chunk_text}"}],
)

extraction = next(b for b in response.content if b.type == "tool_use").input
print(extraction)
# This ONE chunk generates its own LLM call. A real corpus is thousands
# of chunks -- this is the upfront cost from Level 3, point 1, made
# concrete: extraction isn't a side effect, it's a full LLM pass over
# every chunk in the document set, before any user has asked a question.

## Interview-ready summary

**Q: "When would you use GraphRAG instead of standard vector/hybrid retrieval?"**

Weak answer: "GraphRAG is more advanced, so it gives better answers."

Strong answer: "Not universally better -- it's suited to a different question shape. Standard retrieval, like the FAISS+BM25 hybrid setup in my RAG project, answers single-fact lookups against isolated chunks well and cheaply. GraphRAG is built for multi-hop questions that require connecting facts across documents that were never near each other in the source text -- it does this by building a knowledge graph of entities and relationships ahead of time, which requires an LLM extraction pass over the entire corpus, a real and substantial upfront cost compared to standard embedding-based indexing. I'd only reach for it if the actual query patterns genuinely need multi-hop reasoning across entities -- for most single-document-lookup use cases, it's added complexity and cost without a matching benefit, and it introduces its own failure mode: extraction errors silently propagate into every downstream query that relies on the graph structure being correct."